# 엠베딩

In [ ]:
!pip install --q ipython-autotime
%load_ext autotime

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense
from tensorflow.keras.utils import to_categorical


In [ ]:
corpus = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?',
]

labels = [0, 1, 2, 0]  # 각 문장의 카테고리 (0: 첫 번째, 1: 두 번째, 2: 세 번째)

In [ ]:
# 2. Tokenizer로 텍스트 토큰화
vocab_size = 100  # 최대 단어 개수
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(corpus)

In [ ]:
# 단어를 숫자 시퀀스로 변환
sequences = tokenizer.texts_to_sequences(corpus)

# 패딩
maxlen = 6  # 최대 시퀀스 길이
padded = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')


In [ ]:
# 3. 라벨을 원-핫 인코딩
num_classes = len(set(labels))  # 클래스 개수
labels_categorical = to_categorical(labels, num_classes=num_classes)


In [ ]:
# 4. 모델 정의
embedding_dim = 8  # 임베딩 벡터 크기


In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen),
    Flatten(),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# 5. 모델 학습
model.fit(padded, labels_categorical, epochs=10, verbose=1)


In [ ]:
# 6. 새로운 문장 예측
new_sentences = [
    'This is a new document.',
    'Is this the second one?',
    'This document is completely new.'
]
new_sequences = tokenizer.texts_to_sequences(new_sentences)
new_padded = pad_sequences(new_sequences, maxlen=maxlen, padding='post', truncating='post')

predictions = model.predict(new_padded)


In [ ]:
for i, sentence in enumerate(new_sentences):
    print(f"문장: '{sentence}' -> 예측된 카테고리: {np.argmax(predictions[i])}")